## Querying PostgreSQL via SDK

### Installing Utilities and Libraries

In [1]:
%pip install psycopg[binary]==3.3.4 psycopg_pool==3.3.1 python-dotenv

   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.7 MB 3.2 MB/s eta 0:00:01
   ----------------- ---------------------- 1.6/3.7 MB 4.1 MB/s eta 0:00:01
   ------------------------------- -------- 2.9/3.7 MB 4.9 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 5.0 MB/s  0:00:00

   ---------------------------------------- 0/4 [tzdata]
   ---------------------------------------- 0/4 [tzdata]
   ---------------------------------------- 0/4 [tzdata]
   -------------------- ------------------- 2/4 [psycopg-binary]
   ------------------------------ --------- 3/4 [psycopg]
   ------------------------------ --------- 3/4 [psycopg]
   ------------------------------ --------- 3/4 [psycopg]
   ------------------------------ --------- 3/4 [psycopg]
   ---------------------------------------- 4/4 [psycopg]

Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
host = os.getenv("DATABASE_HOSTNAME")
db_name = os.getenv("DATABASE_NAME")
username = os.getenv("DATABASE_USERNAME")
password = os.getenv("DATABASE_PASSWORD")

### Create a Connection Pool

In [4]:
from psycopg_pool import ConnectionPool

pool = ConnectionPool(
    conninfo=(
        f"host={host} "
        f"dbname={db_name} "
        f"user={username} "
        f"password={password} "
        f"sslmode=require"
    ),
    min_size=2,
    max_size=10
)

pool.wait()

print("Connection pool created successfully")

Connection pool created successfully


### Use the Connection Pool to Execute a SELECT Query

In [5]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT *
            FROM ESG.Companies
        """)

        results = cur.fetchall()

        for row in results:
            print(row)

(1, 'GreenSteel Ltd', 'Manufacturing', 'IN', datetime.datetime(2026, 9, 18, 21, 14, 37, 801105))
(2, 'EcoLogistics Pvt Ltd', 'Transportation', 'IN', datetime.datetime(2026, 9, 18, 21, 14, 37, 801105))
(3, 'SolarGrid Energy', 'Energy', 'US', datetime.datetime(2026, 9, 18, 21, 14, 37, 801105))
(4, 'UrbanRetail Corp', 'Retail', 'UK', datetime.datetime(2026, 9, 18, 21, 14, 37, 801105))


### Filter Data Using WHERE

In [6]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT *
        FROM ESG.EmissionRecords
        WHERE Scope = %s
        """,
        (1,))

        results = cur.fetchall()

        for row in results:
            print(row)

(1, 1, datetime.date(2025, 1, 15), 1, Decimal('1200.50'), 'Furnace Combustion')
(3, 2, datetime.date(2025, 1, 10), 1, Decimal('450.25'), 'Fleet Fuel')


### Filter Emissions Greater than 500

In [7]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
        SELECT *
        FROM ESG.EmissionRecords
        WHERE CO2_Emissions > %s
        """,
        (500,))

        results = cur.fetchall()

        for row in results:
            print(row)

(1, 1, datetime.date(2025, 1, 15), 1, Decimal('1200.50'), 'Furnace Combustion')
(2, 1, datetime.date(2025, 1, 20), 2, Decimal('800.75'), 'Electricity Usage')
(7, 4, datetime.date(2025, 1, 8), 2, Decimal('600.60'), 'Store Electricity')


### Sort by Emissions

In [8]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT *
            FROM ESG.EmissionRecords
            ORDER BY CO2_Emissions ASC
        """)

        results = cur.fetchall()

        for row in results:
            print(row)

(6, 3, datetime.date(2025, 1, 18), 3, Decimal('90.00'), 'Supply Chain')
(5, 3, datetime.date(2025, 1, 5), 2, Decimal('150.10'), 'Grid Electricity')
(8, 4, datetime.date(2025, 1, 22), 3, Decimal('250.30'), 'Logistics')
(4, 2, datetime.date(2025, 1, 12), 3, Decimal('300.40'), 'Third-party Transport')
(3, 2, datetime.date(2025, 1, 10), 1, Decimal('450.25'), 'Fleet Fuel')
(7, 4, datetime.date(2025, 1, 8), 2, Decimal('600.60'), 'Store Electricity')
(2, 1, datetime.date(2025, 1, 20), 2, Decimal('800.75'), 'Electricity Usage')
(1, 1, datetime.date(2025, 1, 15), 1, Decimal('1200.50'), 'Furnace Combustion')


### Calculate Total Emissions per Company

In [9]:
with pool.connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                CompanyID,
                SUM(CO2_Emissions) AS TotalEmissions
            FROM ESG.EmissionRecords
            GROUP BY CompanyID
        """)

        results = cur.fetchall()

        for row in results:
            print(row)

(3, Decimal('240.10'))
(4, Decimal('850.90'))
(2, Decimal('750.65'))
(1, Decimal('2001.25'))


### Get Pool Statistics

In [10]:
print(
    pool.get_stats()
)

{'connections_num': 2, 'connections_ms': 1624, 'requests_num': 5, 'usage_ms': 1704, 'pool_min': 2, 'pool_max': 10, 'pool_size': 2, 'pool_available': 2, 'requests_waiting': 0}
